## SABR: a stochastic-vol model that gives a smile formula

SABR stands for **Stochastic Alpha Beta Rho**, after the stochastic volatility
and three of its four parameters ($\alpha$ level, $\beta$ backbone, $\rho$ skew).
The fourth parameter, $\nu$ (vol-of-vol, controlling curvature), is not in the
acronym but is just as essential.

### Why another model, and how it differs from SVI

SVI is a static shape. It fits one slice beautifully but has no dynamics, no
story about how volatility itself moves. SABR (Stochastic Alpha Beta Rho,
Hagan et al. 2002) is a genuine stochastic volatility model: it posits an SDE
for the forward and a second SDE for its volatility, and from those dynamics it
produces a smile. So where SVI *describes* a smile, SABR *generates* one from
assumptions about how the world moves.

The reason SABR is used everywhere in rates and FX desks, despite Heston also
being a stochastic-vol model, is a single practical fact: SABR comes with a
closed-form asymptotic formula for implied volatility. No Fourier inversion, no
PDE solve, no Monte Carlo. You plug the four parameters into an algebraic
expression and get the Black implied vol at any strike directly. That makes
calibration almost trivially fast, which is why it became the market standard
for quoting and interpolating smiles.

### The model

SABR describes the forward $F_t$ and its volatility $\alpha_t$ under the
forward measure:

$$dF_t = \alpha_t \, F_t^{\beta} \, dW_t^F$$
$$d\alpha_t = \nu \, \alpha_t \, dW_t^{\alpha}$$
$$dW_t^F \, dW_t^{\alpha} = \rho \, dt$$

Read each piece:

- $F_t$ is the forward (not spot), which is why your parity-implied forward is
  exactly the right input. SABR is natively a forward-measure model.
- $\alpha_t$ is the instantaneous volatility, itself stochastic. It follows a
  driftless geometric Brownian motion, so it is a lognormal vol-of-vol process,
  always positive, no mean reversion.
- The four parameters are:
  - $\alpha$ (alpha): the initial volatility level, sets the overall height of
    the smile (roughly the ATM vol).
  - $\beta$ (beta): the backbone exponent, $0 \le \beta \le 1$, controlling how
    the ATM vol moves as the forward moves. It sets the assumed
    forward-vol relationship.
  - $\rho$ (rho): the correlation between forward and vol shocks, sets the
    skew, exactly as in SVI and Heston. Negative $\rho$ steepens the left wing.
  - $\nu$ (nu): the vol-of-vol, how erratic the volatility is, controls the
    curvature (smile convexity). $\nu \to 0$ collapses the smile toward flat.

Four parameters, and they map cleanly onto the four things you can see in a
smile: level ($\alpha$), backbone/skew-vs-level tradeoff ($\beta$), skew
($\rho$), and curvature ($\nu$).

### The role of beta, and why it is usually fixed

$\beta$ is the parameter that makes SABR subtle. It interpolates between two
classic model regimes:

- $\beta = 1$: the forward is lognormal (like Black-Scholes). ATM vol is roughly
  invariant as the forward moves. This is the usual choice for equity and FX.
- $\beta = 0$: the forward is normal (Bachelier / normal model). ATM vol scales
  inversely with the forward. Common for rates, especially when forwards can go
  negative.
- $\beta = 0.5$: the CIR-like square-root regime, sometimes used for rates.

The important practical point: $\beta$ and $\rho$ are badly identified from a
single smile. Both affect the skew, so a fit can trade one off against the other
and land almost anywhere along a ridge. The standard resolution is to *fix*
$\beta$ a priori (from the asset class or from how the ATM vol/forward
relationship behaves historically) and calibrate only $(\alpha, \rho, \nu)$.
This is the same identifiability lesson from SVI, two parameters fighting over
the same observable feature, resolved here by pinning one rather than by a
reparametrization.

### The Hagan implied-vol formula

The value of SABR is that the model above has an asymptotic expansion (small
time to expiry, or equivalently a short-maturity / weak-vol-of-vol limit) for
the Black implied volatility. For a forward $F$, strike $K$, expiry $T$:

$$\sigma_B(K, F) = \frac{\alpha}
{(FK)^{(1-\beta)/2}\left[1 + \frac{(1-\beta)^2}{24}\log^2\frac{F}{K}
+ \frac{(1-\beta)^4}{1920}\log^4\frac{F}{K}\right]}
\cdot \frac{z}{x(z)} \cdot \left[1 + \Big(\cdots\Big) T \right]$$

with

$$z = \frac{\nu}{\alpha}(FK)^{(1-\beta)/2}\log\frac{F}{K}, \qquad
x(z) = \log\!\left(\frac{\sqrt{1 - 2\rho z + z^2} + z - \rho}{1 - \rho}\right)$$

and the $T$-order correction term in brackets:

$$1 + \left[\frac{(1-\beta)^2}{24}\frac{\alpha^2}{(FK)^{1-\beta}}
+ \frac{1}{4}\frac{\rho\beta\nu\alpha}{(FK)^{(1-\beta)/2}}
+ \frac{2 - 3\rho^2}{24}\nu^2\right] T$$

It looks forbidding, but it is just an algebraic function of the four parameters
and $(K, F, T)$. Three structural things to notice, which matter more than the
exact coefficients:

1. The leading factor $\alpha / (FK)^{(1-\beta)/2}$ is the level. At the money it
   reduces to $\alpha / F^{1-\beta}$, the ATM vol, which is why $\alpha$ is
   essentially "the ATM vol knob."

2. The $z / x(z)$ factor carries the skew and is where $\rho$ and $\nu$ enter.
   As $K \to F$ (at the money), $z \to 0$ and $z/x(z) \to 1$, so this factor is
   a pure smile-shape multiplier that switches off at the money.

3. The $[1 + (\cdots)T]$ bracket is the small-time correction. It is a genuine
   approximation term, and it is the source of SABR's known failure mode.

### The ATM special case

At $K = F$ the formula collapses (the $\log(F/K)$ terms vanish) to a clean
expression:

$$\sigma_{ATM} = \frac{\alpha}{F^{1-\beta}}
\left[1 + \left(\frac{(1-\beta)^2}{24}\frac{\alpha^2}{F^{2-2\beta}}
+ \frac{1}{4}\frac{\rho\beta\nu\alpha}{F^{1-\beta}}
+ \frac{2 - 3\rho^2}{24}\nu^2\right)T\right]$$

This is worth implementing separately and testing against, because it is the
$K \to F$ limit of the general formula, and if the general formula does not
converge to this as $K \to F$ you have a bug. It is also numerically important:
the general formula has $\log(F/K)$ in denominators, so evaluating exactly at
$K = F$ divides by zero. The code must branch to the ATM expression near the
money, the same removable-singularity handling you have hit before.

### Where SABR breaks, and why we still use it

The Hagan formula is an *asymptotic approximation*, valid for short maturities
and modest vol-of-vol. It has two well-known failure modes:

- **Long maturities / high vol-of-vol**: the $O(T)$ truncation degrades, and the
  approximation drifts from the true SABR model price.
- **Low strikes**: the formula can produce an implied *density* that goes
  negative in the left wing (a butterfly arbitrage), especially for low $\beta$
  and long $T$. This is the SABR analogue of the Durrleman condition you built
  for SVI, and it is why practitioners either use the model only where it is
  well-behaved, or switch to arbitrage-free variants (Hagan's own 2014
  arbitrage-free PDE, or Antonov's exact formulas).

So SABR occupies a different point in the tradeoff space than SVI or Heston. SVI
is arbitrage-free by construction (once you enforce $g \ge 0$) but has no
dynamics. Heston has full dynamics and is arbitrage-free but needs Fourier
pricing. SABR has dynamics *and* a closed-form smile, at the cost of being an
approximation that can violate arbitrage in the wings. For calibration speed and
market-standard quoting it wins; for a guaranteed-clean surface it does not.

### What we will build

The calibration mirrors the SVI structure closely, which is the point of doing
it second:

1. `sabr_vol(K, F, T, alpha, beta, rho, nu)`: the Hagan formula, with a branch
   to the ATM expression near $K = F$.
2. Validation: confirm the general formula converges to the ATM formula as
   $K \to F$ (the removable singularity), and that a flat smile emerges as
   $\nu \to 0$.
3. Fix $\beta$ (equity convention $\beta = 1$, or test a couple of values), then
   calibrate $(\alpha, \rho, \nu)$ to the same CBOE slice we fit with SVI, by
   least squares on implied vols.
4. Overlay the SABR fit on the SVI fit for the same slice, and compare: two
   different philosophies fitting the same market, one a static shape, one a
   dynamic model.